# Scenario Schema Consistency Test

Run Scenario Schema **3 times on the same origin date** and compare:
- Scenario design and structure
- Price point forecasts
- Full price distributions (quantiles)
- Rationales

This reveals whether the agent's estimates are stable or show significant variance across runs.

## Setup

In [1]:
import pandas as pd
import numpy as np
from datetime import datetime
from aieng.forecasting.models import LITE_MODEL
from aieng.forecasting.evaluation.prediction import ContinuousForecast
from energy_oil_forecasting.data import WTI_SERIES_ID, build_wti_multivariate_service
from energy_oil_forecasting.scenario_schema_enhanced import (
    build_wti_news_scenario_schema_enhanced_config,
    build_wti_scenario_schema_enhanced_predictor,
)

# Configuration
ORIGIN_DATE = pd.Timestamp("2026-05-25")  # Change this to test different dates
NUM_RUNS = 3
HORIZONS = [5, 10, 21]

# Setup
data_service = build_wti_multivariate_service()
print(f"Testing Scenario Schema on origin: {ORIGIN_DATE.date()}")
print(f"Horizons: {HORIZONS} business days")
print(f"Number of runs: {NUM_RUNS}")

Testing Scenario Schema on origin: 2026-05-25
Horizons: [5, 10, 21] business days
Number of runs: 3


## Run Scenario Schema 3 Times

In [2]:
from aieng.forecasting.evaluation import MultiTargetBacktestSpec, cached_multi_backtest

results = []

for run_num in range(NUM_RUNS):
    print(f"\n{'='*72}")
    print(f"RUN {run_num + 1} / {NUM_RUNS}")
    print(f"{'='*72}")
    
    # Build predictor
    config = build_wti_news_scenario_schema_enhanced_config(model=LITE_MODEL)
    predictor = build_wti_scenario_schema_enhanced_predictor(config)
    
    # Create minimal spec for this one origin
    spec = MultiTargetBacktestSpec(
        spec_id="scenario_schema_enhanced_consistency_test",
        start=ORIGIN_DATE.strftime("%Y-%m-%d"),
        end=(ORIGIN_DATE + pd.Timedelta(days=1)).strftime("%Y-%m-%d"),
        stride=1,
        tasks=[{
            "task_id": "wti_forecast",
            "description": "WTI price forecast",
            "target_series_id": WTI_SERIES_ID,
            "frequency": "B",
            "horizons": HORIZONS,
        }],
    )
    
    # Run backtest
    backtest_result = cached_multi_backtest(
        predictor, spec, data_service,
        max_retries=3,
        force_refresh=True,  # Force fresh run each time
    )
    
    results.append({
        "run_num": run_num + 1,
        "backtest_result": backtest_result,
    })
    
    # Extract predictions
    if backtest_result:
        result = next(iter(backtest_result.values()))
        for pred in result.predictions:
            if isinstance(pred.payload, ContinuousForecast):
                print(f"  point=${pred.payload.point_forecast:.2f}")

print(f"\n✓ All {NUM_RUNS} runs complete")


RUN 1 / 3


/home/coder/agentic-forecasting/.venv/lib/python3.12/site-packages/google/adk/tools/function_tool.py:95: UserWarning: [EXPERIMENTAL] feature FeatureName.JSON_SCHEMA_FOR_FUNC_DECL is enabled.
  build_function_declaration(


  point=$94.50
  point=$92.00
  point=$88.00
  point=$97.50
  point=$98.50
  point=$100.00

RUN 2 / 3
  point=$97.50
  point=$99.00
  point=$102.00
  point=$97.50
  point=$99.00
  point=$102.00

RUN 3 / 3
  point=$97.50
  point=$99.00
  point=$102.00
  point=$97.50
  point=$98.00
  point=$99.00

✓ All 3 runs complete


In [3]:
print("="*72)
print("FACTORS AND SCENARIOS COMPARISON ACROSS RUNS")
print("="*72)
print(f"Testing Scenario Schema on origin: {ORIGIN_DATE.date()}")

for run_data in results:
    run_num = run_data["run_num"]
    backtest_result = run_data["backtest_result"]
    
    if not backtest_result:
        continue
    
    result = next(iter(backtest_result.values()))
    pred = result.predictions[0] if result.predictions else None
    
    if not pred or not pred.metadata:
        continue
    
    metadata = pred.metadata
    factors = metadata.get("factors", [])
    scenarios = metadata.get("scenarios", [])
    rationale = metadata.get("rationale", "")
    
    print(f"\n{'─'*72}")
    print(f"RUN {run_num}")
    print(f"{'─'*72}")
    
    # Display factors
    print(f"\nFACTORS ({len(factors)}):")
    for factor in factors:
        name = factor.get('name')
        tier = factor.get('tier')
        impact = factor.get('impact_score', 'N/A')
        impact_str = f", impact={impact}" if tier == "transitory" else ""
        print(f"  • {name} (tier={tier}{impact_str})")
    
    # Display scenarios
    print(f"\nSCENARIOS ({len(scenarios)}):")
    for scenario in scenarios:
        name = scenario.get('name')
        prob = scenario.get('probability')
        low = scenario.get('price_low')
        high = scenario.get('price_high')
        tail = scenario.get('is_tail_case')
        tail_str = " [TAIL]" if tail else ""
        print(f"  • {name}{tail_str}: P={prob:.1%}, Price ${low:.2f}–${high:.2f}")
    
    # Display rationale
    print(f"\nRATIONALE:\n  {rationale}")

FACTORS AND SCENARIOS COMPARISON ACROSS RUNS
Testing Scenario Schema on origin: 2026-05-25

────────────────────────────────────────────────────────────────────────
RUN 1
────────────────────────────────────────────────────────────────────────

FACTORS (3):
  • Strait of Hormuz Supply Disruption (tier=core)
  • Global Demand Destruction (tier=core)
  • Diplomatic Negotiation Progress (tier=transitory, impact=high)

SCENARIOS (3):
  • Diplomatic Breakthrough: P=45.0%, Price $75.00–$85.00
  • Stalemate and Escalation: P=40.0%, Price $95.00–$115.00
  • Total Regional Conflict (Tail Case) [TAIL]: P=15.0%, Price $120.00–$150.00

RATIONALE:
  The market is currently in a high-stakes transition phase. The primary driver is the Strait of Hormuz crisis, which has created a massive risk premium. The recent price decline reflects market anticipation of a diplomatic resolution. My forecast assumes a base case where the market continues to price in a gradual reduction of this premium, while acknowl

## Compare Point Forecasts Across Runs

In [4]:
# Extract point forecasts for each horizon across all runs
forecast_comparison = {h: [] for h in HORIZONS}

for run_data in results:
    backtest_result = run_data["backtest_result"]
    
    # backtest_result is a dict of results; extract from the first one
    for result in backtest_result.values():
        for pred in result.predictions:
            if isinstance(pred.payload, ContinuousForecast):
                # Find which horizon this is
                as_of = pd.Timestamp(pred.as_of)
                forecast_date = pd.Timestamp(pred.forecast_date)
                offset = pd.tseries.offsets.BDay()
                
                for h in HORIZONS:
                    target_date = as_of + offset * h
                    if target_date.normalize() == forecast_date.normalize():
                        forecast_comparison[h].append(pred.payload.point_forecast)
                        break

# Display comparison table
comparison_rows = []
for h in HORIZONS:
    forecasts = forecast_comparison[h]
    if forecasts:
        row = {
            "Horizon": f"{h}d",
            "Run 1": f"${forecasts[0]:.2f}" if len(forecasts) > 0 else "—",
            "Run 2": f"${forecasts[1]:.2f}" if len(forecasts) > 1 else "—",
            "Run 3": f"${forecasts[2]:.2f}" if len(forecasts) > 2 else "—",
            "Mean": f"${np.mean(forecasts):.2f}",
            "Std Dev": f"${np.std(forecasts):.2f}",
            "Range": f"${np.max(forecasts) - np.min(forecasts):.2f}",
        }
        comparison_rows.append(row)

df_comparison = pd.DataFrame(comparison_rows)
print("\n" + "="*72)
print("POINT FORECAST COMPARISON ACROSS 3 RUNS")
print("="*72)
print(df_comparison.to_string(index=False))


POINT FORECAST COMPARISON ACROSS 3 RUNS
Horizon  Run 1   Run 2   Run 3   Mean Std Dev  Range
     5d $94.50  $97.50  $97.50 $97.00   $1.12  $3.00
    10d $92.00  $98.50  $99.00 $97.58   $2.52  $7.00
    21d $88.00 $100.00 $102.00 $98.83   $4.98 $14.00


## Extract Full Distributions (Quantiles)

In [5]:
# Extract full distributions and build comparison table
all_distributions = {run_idx: {h: None for h in HORIZONS} for run_idx in range(NUM_RUNS)}

for run_idx, run_data in enumerate(results):
    backtest_result = run_data["backtest_result"]
    
    for result in backtest_result.values():
        for pred in result.predictions:
            if isinstance(pred.payload, ContinuousForecast):
                cf = pred.payload
                as_of = pd.Timestamp(pred.as_of)
                forecast_date = pd.Timestamp(pred.forecast_date)
                offset = pd.tseries.offsets.BDay()
                
                for h in HORIZONS:
                    target_date = as_of + offset * h
                    if target_date.normalize() == forecast_date.normalize():
                        if all_distributions[run_idx][h] is None:
                            all_distributions[run_idx][h] = cf
                        break

# Build table
dist_rows = []
for h in HORIZONS:
    row = {"Horizon": f"{h}d"}
    for run_idx in range(NUM_RUNS):
        cf = all_distributions[run_idx][h]
        if cf is not None:
            point = f"${cf.point_forecast:.2f}"
            
            # Try to get CI
            ci = ""
            if hasattr(cf, 'lower_quantile') and hasattr(cf, 'upper_quantile'):
                ci = f" [{cf.lower_quantile:.2f}, {cf.upper_quantile:.2f}]"
            elif hasattr(cf, 'quantile_forecasts') and cf.quantile_forecasts:
                q10 = cf.quantile_forecasts.get(0.1, "—")
                q90 = cf.quantile_forecasts.get(0.9, "—")
                ci = f" [{q10:.2f}, {q90:.2f}]"
            
            row[f"Run {run_idx+1}"] = point + ci
    dist_rows.append(row)

df_distributions = pd.DataFrame(dist_rows)
print("\n" + "="*72)
print("FULL DISTRIBUTIONS BY HORIZON (point + 80% CI)")
print("="*72)
print(df_distributions.to_string(index=False))


FULL DISTRIBUTIONS BY HORIZON (point + 80% CI)
Horizon  Run 1   Run 2   Run 3
     5d $94.50  $97.50  $97.50
    10d $92.00  $99.00  $99.00
    21d $88.00 $102.00 $102.00


## Extract Rationales (Agent Reasoning)

In [6]:
# Extract rationales from prediction metadata - deduplicated
for run_idx, run_data in enumerate(results):
    print(f"\n{'='*72}")
    print(f"RUN {run_idx + 1} — AGENT RATIONALE")
    print(f"{'='*72}\n")
    
    backtest_result = run_data["backtest_result"]
    seen_rationales = set()
    
    # Extract from the first result in the dict
    for result in backtest_result.values():
        for pred in result.predictions:
            # Check for rationale in metadata
            if hasattr(pred, 'metadata') and pred.metadata:
                rationale = pred.metadata.get('rationale', None)
                if rationale and rationale not in seen_rationales:
                    print(rationale)
                    print()
                    seen_rationales.add(rationale)
            
            # Also check payload for any reasoning field
            if hasattr(pred.payload, 'reasoning'):
                reasoning = pred.payload.reasoning
                if reasoning and reasoning not in seen_rationales:
                    print(reasoning)
                    print()
                    seen_rationales.add(reasoning)


RUN 1 — AGENT RATIONALE

The market is currently in a high-stakes transition phase. The primary driver is the Strait of Hormuz crisis, which has created a massive risk premium. The recent price decline reflects market anticipation of a diplomatic resolution. My forecast assumes a base case where the market continues to price in a gradual reduction of this premium, while acknowledging the significant tail risk of a total conflict escalation.

The market is currently in a high-volatility regime, reflecting a tug-of-war between supply-side constraints (OPEC+ and geopolitical risks) and concerns over global demand sustainability. The forecast reflects a cautious bias toward the upside in the near term, with significant tail risk if economic indicators deteriorate.


RUN 2 — AGENT RATIONALE

The market is currently dominated by the supply shock from the Strait of Hormuz. The primary uncertainty is the duration of this disruption. My forecast reflects a base case of a continued impasse, wit

## Consistency Assessment

In [7]:
print("\n" + "="*72)
print("CONSISTENCY METRICS")
print("="*72)

for h in HORIZONS:
    forecasts = forecast_comparison[h]
    if len(forecasts) == NUM_RUNS:
        mean = np.mean(forecasts)
        std = np.std(forecasts)
        cv = (std / mean) * 100
        print(f"\nh={h}d:")
        print(f"  Mean:                 ${mean:.2f}")
        print(f"  Std Dev:              ${std:.2f}")
        print(f"  Coefficient of Var:   {cv:.1f}%")
        print(f"  Range (max - min):    ${np.max(forecasts) - np.min(forecasts):.2f}")
        
        if cv < 1.0:
            consistency = "✓ Very consistent (CV < 1%)" 
        elif cv < 2.0:
            consistency = "✓ Consistent (CV 1-2%)"
        elif cv < 5.0:
            consistency = "⚠ Moderate variance (CV 2-5%)"
        else:
            consistency = "✗ High variance (CV > 5%)"
        print(f"  Assessment:           {consistency}")


CONSISTENCY METRICS


## Scenario Structure Comparison

**Scenarios identified in Run 1:**
- Scenario A: ...
- Scenario B: ...
- Scenario C: ...

**Scenarios identified in Run 2:**
- Scenario A: ...
- Scenario B: ...
- Scenario C: ...

**Scenarios identified in Run 3:**
- Scenario A: ...
- Scenario B: ...
- Scenario C: ...

**Consistency of scenarios:**
- Same three scenarios across all runs?
- Same base-case probability weighting?
- Same tail structure (upside vs downside)?

In [8]:
import json

# Extract json_response from prediction metadata
for run_idx, run_data in enumerate(results):
    print(f"\n{'='*72}")
    print(f"RUN {run_idx + 1} — JSON RESPONSE (scenarios + factors)")
    print(f"{'='*72}\n")
    
    backtest_result = run_data["backtest_result"]
    seen_responses = set()
    
    for result in backtest_result.values():
        for pred in result.predictions:
            if hasattr(pred, 'metadata') and pred.metadata:
                json_response = pred.metadata.get('json_response', None)
                if json_response:
                    # json_response might be a string, dict, or already parsed
                    try:
                        if isinstance(json_response, str):
                            parsed = json.loads(json_response)
                        else:
                            parsed = json_response
                        
                        response_str = json.dumps(parsed, indent=2)
                        if response_str not in seen_responses:
                            print(response_str)
                            print()
                            seen_responses.add(response_str)
                    except json.JSONDecodeError as e:
                        print(f"Failed to parse json_response: {e}")
                        print(json_response)
                        print()


RUN 1 — JSON RESPONSE (scenarios + factors)


RUN 2 — JSON RESPONSE (scenarios + factors)


RUN 3 — JSON RESPONSE (scenarios + factors)

